# Notebook 6: Batch Train All Continual SD-LoRA Adapters

Bu notebook, Notebook 2 akisini baz alir ve tanimli adapterleri sirayla egitir.

Akis:
1. Repo bootstrap ve Notebook 2 helper'larini yukle.
2. Adapter matrisi ve sirayi tanimla.
3. Her adapter icin Notebook 2 parametre, dataset validation, training, OOD calibration, export ve final evaluation hucrelerini calistir.
4. Her run icin ozet yazdir; runtime auto-disconnect batch bitene kadar kapali kalir.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from urllib.parse import urlsplit, urlunsplit

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
NOTEBOOK6_SPARSE_PATHS = (
    'README.md',
    'docs',
    'src',
    'scripts',
    'config',
    'colab_notebooks',
    'requirements.txt',
    'requirements_colab.txt',
    'pyproject.toml',
)


def _build_repo_access_url(repo_url, token):
    token = str(token or '').strip()
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))


def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    clone_url = _build_repo_access_url(REPO_URL, os.environ.get('GH_TOKEN') or os.environ.get('GITHUB_TOKEN'))
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', clone_url, str(CLONE_TARGET)], check=True)
        subprocess.run(['git', 'sparse-checkout', 'set', *NOTEBOOK6_SPARSE_PATHS], cwd=str(CLONE_TARGET), check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET


_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell01_bootstrap_access.py', globals())


In [ ]:
NOTEBOOK_NAME = "6_train_all_continual_sd_lora_adapters.ipynb"
NOTEBOOK_FILENAME = "6_train_all_continual_sd_lora_adapters.executed.ipynb"
AUTO_DISCONNECT_RUNTIME = False
AUTO_PUSH_TO_GITHUB = True
NB6_AUTO_DISCONNECT_RUNTIME = True
NB6_AUTO_DISCONNECT_GRACE_SECONDS = 20
ENABLE_BAYESIAN_OPTIMIZATION = True
MANUAL_PARAM_OVERRIDES = {}
DEFAULT_RUNTIME_PARAMS = {
    "AUTO_DISCONNECT_RUNTIME": False,
    "AUTO_PUSH_TO_GITHUB": True,
}
NB6_MANUAL_PARAM_OVERRIDES = {}

from scripts.notebook_helpers.adapter_recommendations import get_adapter_recs
ADAPTER_RECS = get_adapter_recs()

NB6_ADAPTER_SEQUENCE = [
    "apricot__leaf",
    "apricot__fruit",
    "strawberry__leaf",
    "strawberry__fruit",
    "grape__leaf",
    "grape__fruit",
    "tomato__leaf",
    "tomato__fruit",
]


In [ ]:
import json

NB6_RESULTS = {}

for index, adapter_key in enumerate(NB6_ADAPTER_SEQUENCE, start=1):
    print(f"\n[NB6] Starting {index}/{len(NB6_ADAPTER_SEQUENCE)}: {adapter_key}")
    adapter_rec = ADAPTER_RECS[adapter_key]
    CROP_NAME = adapter_rec["crop"]
    PART_NAME = adapter_rec["part"]
    DATASET_NAME = adapter_key
    ADAPTER_KEY = adapter_key
    MANUAL_PARAM_OVERRIDES = dict(NB6_MANUAL_PARAM_OVERRIDES.get(adapter_key, {}))
    try:
        run_cell_script('nb2_cell03_runtime_setup.py', globals())
        run_cell_script('nb2_cell04_parameter_resolution.py', globals())
        run_cell_script('nb2_cell05_access_check.py', globals())
        run_cell_script('nb2_cell06_dataset_validation.py', globals())
        run_cell_script('nb2_cell07_engine_init.py', globals())
        run_cell_script('nb2_cell08_ood_config_verify.py', globals())
        run_cell_script('nb2_cell09_training.py', globals())
        run_cell_script('nb2_cell10_ood_calibration.py', globals())
        run_cell_script('nb2_cell11_adapter_save.py', globals())
        run_cell_script('nb2_cell12_final_evaluation.py', globals())
        NB6_RESULTS[adapter_key] = {
            "status": "ok",
            "run_id": str(globals().get("RUN_ID", "")),
            "crop": str(globals().get("CROP_NAME", "")),
            "part": str(globals().get("PART_NAME", "")),
        }
    except Exception as exc:
        NB6_RESULTS[adapter_key] = {
            "status": "failed",
            "error": str(exc),
            "run_id": str(globals().get("RUN_ID", "")),
            "crop": str(globals().get("CROP_NAME", "")),
            "part": str(globals().get("PART_NAME", "")),
        }
        print(f"[NB6] FAILED {adapter_key}: {exc}")
        continue

print('\n[NB6] SUMMARY')
print(json.dumps(NB6_RESULTS, indent=2, ensure_ascii=False))

from scripts.colab_notebook_helpers import maybe_auto_disconnect_colab_runtime

NB6_COMPLETION_REPORT = {
    "ready": True,
    "checks": {
        "batch_loop_completed": True,
        "all_adapters_attempted": len(NB6_RESULTS) == len(NB6_ADAPTER_SEQUENCE),
    },
    "missing": [],
    "soft_missing": [],
    "adapter_results": NB6_RESULTS,
}
maybe_auto_disconnect_colab_runtime(
    enabled=bool(NB6_AUTO_DISCONNECT_RUNTIME),
    grace_period_sec=float(NB6_AUTO_DISCONNECT_GRACE_SECONDS),
    telemetry=globals().get("TELEMETRY"),
    completion_report=NB6_COMPLETION_REPORT,
)
